In [15]:
import pandas as pd 

import numpy as np

df=pd.read_csv("./Nuclear_Power_Plant_CPS_Dataset.csv")
df.head()

,Timestamp,Reactor Temp (°C),Coolant Flow Rate (L/s),Pressure (MPa),Radiation Level (μSv/h),Turbine Speed (RPM),Pump Status,Power Output (MW),Control Rod Position (%),Steam Flow Rate (kg/s),Vibration Level (mm/s),Water Level (m),Anomaly Detected
0,2024-10-16 10:00:00,357.450712,471.878368,15.639131,0.352819,3301.404054,ON,978.518251,61.914141,220.491605,1.090474,5.077447,No
1,2024-10-16 10:01:00,347.926035,463.558917,16.088410,0.384267,3188.520927,ON,934.944537,71.731564,220.318340,2.934264,5.213480,No
2,2024-10-16 10:02:00,359.715328,471.834463,15.726660,1.214944,2920.910299,ON,978.550964,66.849758,205.981092,1.303091,5.198166,Yes
3,2024-10-16 10:03:00,372.845448,478.149470,15.864172,0.600205,3058.417812,ON,1048.238495,66.329191,221.163781,2.092060,5.188259,No
4,2024-10-16 10:04:00,346.487699,470.732901,16.836726,0.834639,3147.450650,OFF,1064.999369,62.767509,240.951403,2.034543,5.189138,No


In [16]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 13 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Timestamp                 1200 non-null   str    
 1   Reactor Temp (°C)         1200 non-null   float64
 2   Coolant Flow Rate (L/s)   1200 non-null   float64
 3   Pressure (MPa)            1200 non-null   float64
 4   Radiation Level (μSv/h)   1200 non-null   float64
 5   Turbine Speed (RPM)       1200 non-null   float64
 6   Pump Status               1200 non-null   str    
 7   Power Output (MW)         1200 non-null   float64
 8   Control Rod Position (%)  1200 non-null   float64
 9   Steam Flow Rate (kg/s)    1200 non-null   float64
 10  Vibration Level (mm/s)    1200 non-null   float64
 11  Water Level (m)           1200 non-null   float64
 12  Anomaly Detected          1200 non-null   str    
dtypes: float64(10), str(3)
memory usage: 122.0 KB


In [17]:
df.describe()

,Reactor Temp (°C),Coolant Flow Rate (L/s),Pressure (MPa),Radiation Level (μSv/h),Turbine Speed (RPM),Power Output (MW),Control Rod Position (%),Steam Flow Rate (kg/s),Vibration Level (mm/s),Water Level (m)
count,1200.000000,1200.000000,1200.000000,1200.000000,1200.000000,1200.000000,1200.000000,1200.000000,1200.000000,1200.000000
mean,350.575733,470.369775,16.008382,0.581900,3192.985499,1000.442858,70.123696,219.841024,1.990574,5.198914
std,14.825930,14.756203,0.511526,0.295586,201.808719,52.127700,4.996608,14.363382,0.500757,0.098022
min,301.380990,424.707318,14.504432,-0.353011,2620.097224,815.581735,50.816722,161.163996,0.312210,4.896601
25%,340.624931,460.370079,15.646462,0.386899,3061.261439,965.441255,66.845331,210.077114,1.661971,5.135156
50%,350.724549,470.184693,16.008505,0.593313,3192.707942,999.437283,70.100292,220.139668,1.975637,5.199864
75%,360.140211,480.089684,16.346386,0.777762,3326.666520,1036.376938,73.409052,229.360633,2.318087,5.264305
max,407.790972,517.896614,17.963119,1.572928,3822.582040,1155.884056,86.886915,263.711638,3.688884,5.528412


In [18]:
# Remove the 'Timestamp' column as it's not directly used for classification in its raw form.
# If time-series analysis was required, we would extract features from it (e.g., hour, day of week).
# Re-running this cell to ensure preprocessing is applied to the correct 'df' after initial load.
if 'Timestamp' in df.columns:
    df = df.drop('Timestamp', axis=1)
    print("\n'Timestamp' column removed.")
else:
    print("\n'Timestamp' column not found or already removed.")

# Identify categorical columns (excluding the target column)
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
if 'Anomaly Detected' in categorical_cols:
    categorical_cols.remove('Anomaly Detected')

print(f"\nCategorical columns identified for one-hot encoding: {categorical_cols}")

# Handle categorical features using one-hot encoding
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)
print("\nCategorical features (e.g., Pump Status) have been one-hot encoded.")

# Encode the target column 'Anomaly Detected': Yes=1, No=0
df['Anomaly Detected'] = df['Anomaly Detected'].map({'Yes': 1, 'No': 0})
print("\nTarget column 'Anomaly Detected' encoded (Yes=1, No=0).")

print("\nDataset after preprocessing:")
display(df.head())

# Split the data into features and target
X = df.drop('Anomaly Detected', axis=1)
y = df['Anomaly Detected']

# First split into training+validation and holdout test set
from sklearn.model_selection import train_test_split

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Then split the training+validation set into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, random_state=42, stratify=y_train_val
)
# 0.25 x 0.80 = 0.20 of the full dataset for validation

print(f"\nTrain set shape: {X_train.shape}")
print(f"Validation set shape: {X_val.shape}")
print(f"Test set shape: {X_test.shape}")

# Compare multiple classifiers on validation, test, and full dataset
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, roc_auc_score

models = [
    ('Logistic Regression', Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=2000, random_state=42))
    ])),
    ('Decision Tree', DecisionTreeClassifier(random_state=42)),
    ('Random Forest', RandomForestClassifier(n_estimators=100, random_state=42)),
    ('Gradient Boosting', GradientBoostingClassifier(random_state=42)),
    ('K-Nearest Neighbors', Pipeline([
        ('scaler', StandardScaler()),
        ('clf', KNeighborsClassifier(n_neighbors=5))
    ])),
    ('Support Vector Machine', Pipeline([
        ('scaler', StandardScaler()),
        ('clf', SVC(probability=True, random_state=42))
    ])),
    ('Gaussian Naive Bayes', GaussianNB())
]

# Add XGBoost if available
try:
    from xgboost import XGBClassifier
    models.append(('XGBoost', XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)))
except Exception:
    print("XGBoost is not installed, skipping it.")

results = []
for name, clf in models:
    # Train the current model on the training set only
    clf.fit(X_train, y_train)

    # Predict on validation set to tune and compare models
    val_pred = clf.predict(X_val)

    # Predict on holdout test set to evaluate generalization
    test_pred = clf.predict(X_test)

    # Predict on the full dataset as requested for a final overall score
    full_pred = clf.predict(X)

    row = {
        'Model': name,
        'Validation Accuracy': accuracy_score(y_val, val_pred),
        'Test Accuracy': accuracy_score(y_test, test_pred),
        'Full Accuracy': accuracy_score(y, full_pred)
    }

    # Compute validation ROC AUC when probability scores are available
    try:
        if hasattr(clf, 'predict_proba'):
            val_prob = clf.predict_proba(X_val)[:, 1]
            row['Validation ROC AUC'] = roc_auc_score(y_val, val_prob)
        elif hasattr(clf, 'decision_function'):
            val_score = clf.decision_function(X_val)
            row['Validation ROC AUC'] = roc_auc_score(y_val, val_score)
    except Exception:
        row['Validation ROC AUC'] = None

    results.append(row)

# Sort models by validation accuracy to compare performance cleanly
results_df = pd.DataFrame(results).sort_values(by='Validation Accuracy', ascending=False)
print("\nModel comparison results:")
print(results_df.round(4).to_string(index=False))

# Show the best model according to validation accuracy
best_model_name = results_df.iloc[0]['Model']
print(f"\nBest model by validation accuracy: {best_model_name}")


'Timestamp' column removed.

Categorical columns identified for one-hot encoding: ['Pump Status']

Categorical features (e.g., Pump Status) have been one-hot encoded.

Target column 'Anomaly Detected' encoded (Yes=1, No=0).

Dataset after preprocessing:


C:\Users\XPS\AppData\Local\Temp\ipykernel_26064\211460644.py:11: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include=['object']).columns.tolist()


,Reactor Temp (°C),Coolant Flow Rate (L/s),Pressure (MPa),Radiation Level (μSv/h),Turbine Speed (RPM),Power Output (MW),Control Rod Position (%),Steam Flow Rate (kg/s),Vibration Level (mm/s),Water Level (m),Anomaly Detected,Pump Status_ON
0,357.450712,471.878368,15.639131,0.352819,3301.404054,978.518251,61.914141,220.491605,1.090474,5.077447,0,True
1,347.926035,463.558917,16.088410,0.384267,3188.520927,934.944537,71.731564,220.318340,2.934264,5.213480,0,True
2,359.715328,471.834463,15.726660,1.214944,2920.910299,978.550964,66.849758,205.981092,1.303091,5.198166,1,True
3,372.845448,478.149470,15.864172,0.600205,3058.417812,1048.238495,66.329191,221.163781,2.092060,5.188259,0,True
4,346.487699,470.732901,16.836726,0.834639,3147.450650,1064.999369,62.767509,240.951403,2.034543,5.189138,0,False



Train set shape: (720, 11)
Validation set shape: (240, 11)
Test set shape: (240, 11)

Model comparison results:
                 Model  Validation Accuracy  Test Accuracy  Full Accuracy  Validation ROC AUC
   Logistic Regression               0.9500         0.9500         0.9483              0.3556
         Random Forest               0.9500         0.9500         0.9800              0.4285
Support Vector Machine               0.9500         0.9500         0.9483              0.6524
   K-Nearest Neighbors               0.9500         0.9500         0.9492              0.4306
  Gaussian Naive Bayes               0.9500         0.9500         0.9483              0.2873
               XGBoost               0.9458         0.9458         0.9783              0.4733
     Gradient Boosting               0.9333         0.9417         0.9700              0.4295
         Decision Tree               0.8625         0.8708         0.9467              0.4539

Best model by validation accuracy: Logis

c:\Users\XPS\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\xgboost\training.py:200: UserWarning: [07:53:13] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [19]:
#Text cell <dd19c632>
# %% [markdown]
## 1. Data Loading and Initial Inspection

#First, we'll load the dataset using pandas and perform an initial inspection to understand its structure, column types, and basic statistics.

#Code cell <4bbbfe58>
# %% [code]
import pandas as pd
import numpy as np

# Load the dataset
data_path = '/content/Nuclear_Power_Plant_CPS_Dataset.csv'
df = pd.read_csv(data_path)

print("Dataset loaded successfully.")
print("\nFirst 5 rows of the dataset:")
display(df.head())

print("\nDataset Information:")
df.info()

print("\nDescriptive Statistics:")
display(df.describe())

		 #   Column                    Non-Null Count  Dtype  
		---  ------                    --------------  -----  
		 0   Timestamp                 1200 non-null   object 
		 1   Reactor Temp (°C)         1200 non-null   float64
		 2   Coolant Flow Rate (L/s)   1200 non-null   float64
		 3   Pressure (MPa)            1200 non-null   float64
		 4   Radiation Level (μSv/h)   1200 non-null   float64
		 5   Turbine Speed (RPM)       1200 non-null   float64
		 6   Pump Status               1200 non-null   object 
		 7   Power Output (MW)         1200 non-null   float64
		 8   Control Rod Position (%)  1200 non-null   float64
		 9   Steam Flow Rate (kg/s)    1200 non-null   float64
		 10  Vibration Level (mm/s)    1200 non-null   float64
		 11  Water Level (m)           1200 non-null   float64
		 12  Anomaly Detected          1200 non-null   object 
		dtypes: float64(10), object(3)
		memory usage: 122.0+ KB
		
		Descriptive Statistics:
	text/plain
		Timestamp  Reactor Temp (°C)  Coolant Flow Rate (L/s)  \
		0  2024-10-16 10:00:00         357.450712               471.878368   
		1  2024-10-16 10:01:00         347.926035               463.558917   
		2  2024-10-16 10:02:00         359.715328               471.834463   
		3  2024-10-16 10:03:00         372.845448               478.149470   
		4  2024-10-16 10:04:00         346.487699               470.732901   
		
		   Pressure (MPa)  Radiation Level (μSv/h)  Turbine Speed (RPM) Pump Status  \
		0       15.639131                 0.352819          3301.404054          ON   
		1       16.088410                 0.384267          3188.520927          ON   
		2       15.726660                 1.214944          2920.910299          ON   
		3       15.864172                 0.600205          3058.417812          ON   
		4       16.836726                 0.834639          3147.450650         OFF   
		
		   Power Output (MW)  Control Rod Position (%)  Steam Flow Rate (kg/s)  \
		0         978.518251                 61.914141              220.491605   
		1         934.944537                 71.731564              220.318340   
		2         978.550964                 66.849758              205.981092   
		3        1048.238495                 66.329191              221.163781   
		4        1064.999369                 62.767509              240.951403   
		
		   Vibration Level (mm/s)  Water Level (m) Anomaly Detected  
		0                1.090474         5.077447               No  
		1                2.934264         5.213480               No  
		2                1.303091         5.198166              Yes  
		3                2.092060         5.188259               No  
		4                2.034543         5.189138               No
		Reactor Temp (°C)  Coolant Flow Rate (L/s)  Pressure (MPa)  \
		count        1200.000000              1200.000000     1200.000000   
		mean          350.575733               470.369775       16.008382   
		std            14.825930                14.756203        0.511526   
		min           301.380990               424.707318       14.504432   
		25%           340.624931               460.370079       15.646462   
		50%           350.724549               470.184693       16.008505   
		75%           360.140211               480.089684       16.346386   
		max           407.790972               517.896614       17.963119   
		
		       Radiation Level (μSv/h)  Turbine Speed (RPM)  Power Output (MW)  \
		count              1200.000000          1200.000000        1200.000000   
		mean                  0.581900          3192.985499        1000.442858   
		std                   0.295586           201.808719          52.127700   
		min                  -0.353011          2620.097224         815.581735   
		25%                   0.386899          3061.261439         965.441255   
		50%                   0.593313          3192.707942         999.437283   
		75%                   0.777762          3326.666520        1036.376938   
		max                   1.572928          3822.582040        1155.884056   
		
		       Control Rod Position (%)  Steam Flow Rate (kg/s)  \
		count               1200.000000             1200.000000   
		mean                  70.123696              219.841024   
		std                    4.996608               14.363382   
		min                   50.816722              161.163996   
		25%                   66.845331              210.077114   
		50%                   70.100292              220.139668   
		75%                   73.409052              229.360633   
		max                   86.886915              263.711638   
		
		       Vibration Level (mm/s)  Water Level (m)  
		count             1200.000000      1200.000000  
		mean                 1.990574         5.198914  
		std                  0.500757         0.098022  
		min                  0.312210         4.896601  
		25%                  1.661971         5.135156  
		50%                  1.975637         5.199864  
		75%                  2.318087         5.264305  
		max                  3.688884         5.528412

Text cell <69d5b081>
# %% [markdown]
## 2. Preprocessing: Timestamp, Categorical Features, and Target Encoding

We need to process the 'Timestamp' column, handle categorical features like 'Pump Status' using one-hot encoding, and encode the 'Anomaly Detected' target column.

Code cell <2a5dfcfd>
# %% [code]
# Remove the 'Timestamp' column as it's not directly used for classification in its raw form.
# If time-series analysis was required, we would extract features from it (e.g., hour, day of week).
# Re-running this cell to ensure preprocessing is applied to the correct 'df' after initial load.
if 'Timestamp' in df.columns:
    df = df.drop('Timestamp', axis=1)
    print("\n'Timestamp' column removed.")
else:
    print("\n'Timestamp' column not found or already removed.")

# Identify categorical columns (excluding the target column)
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
if 'Anomaly Detected' in categorical_cols:
    categorical_cols.remove('Anomaly Detected')

print(f"\nCategorical columns identified for one-hot encoding: {categorical_cols}")

# Handle categorical features using one-hot encoding
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)
print("\nCategorical features (e.g., Pump Status) have been one-hot encoded.")

# Encode the target column 'Anomaly Detected': Yes=1, No=0
df['Anomaly Detected'] = df['Anomaly Detected'].map({'Yes': 1, 'No': 0})
print("\nTarget column 'Anomaly Detected' encoded (Yes=1, No=0).")

print("\nDataset after preprocessing:")
display(df.head())
Execution output from Jun 14, 2026 9:33 PM
13KB
	Stream
		'Timestamp' column removed.
		
		Categorical columns identified for one-hot encoding: ['Pump Status']
		
		Categorical features (e.g., Pump Status) have been one-hot encoded.
		
		Target column 'Anomaly Detected' encoded (Yes=1, No=0).
		
		Dataset after preprocessing:
	text/plain
		Reactor Temp (°C)  Coolant Flow Rate (L/s)  Pressure (MPa)  \
		0         357.450712               471.878368       15.639131   
		1         347.926035               463.558917       16.088410   
		2         359.715328               471.834463       15.726660   
		3         372.845448               478.149470       15.864172   
		4         346.487699               470.732901       16.836726   
		
		   Radiation Level (μSv/h)  Turbine Speed (RPM)  Power Output (MW)  \
		0                 0.352819          3301.404054         978.518251   
		1                 0.384267          3188.520927         934.944537   
		2                 1.214944          2920.910299         978.550964   
		3                 0.600205          3058.417812        1048.238495   
		4                 0.834639          3147.450650        1064.999369   
		
		   Control Rod Position (%)  Steam Flow Rate (kg/s)  Vibration Level (mm/s)  \
		0                 61.914141              220.491605                1.090474   
		1                 71.731564              220.318340                2.934264   
		2                 66.849758              205.981092                1.303091   
		3                 66.329191              221.163781                2.092060   
		4                 62.767509              240.951403                2.034543   
		
		   Water Level (m)  Anomaly Detected  Pump Status_ON  
		0         5.077447                 0            True  
		1         5.213480                 0            True  
		2         5.198166                 1            True  
		3         5.188259                 0            True  
		4         5.189138                 0           False

Text cell <7573886f>
# %% [markdown]
## 3. Check Missing Values and Data Quality

Before proceeding with EDA and modeling, we'll check for any missing values and ensure data quality.

Code cell <32c9b197>
# %% [code]
# Check for missing values
# Re-running this cell to ensure missing value handling is applied to the correctly preprocessed 'df'.
missing_values = df.isnull().sum()
missing_values = missing_values[missing_values > 0]

if not missing_values.empty:
    print("\nMissing values in each column:")
    display(missing_values)
    # For simplicity, we'll fill missing numerical values with the mean and categorical with mode.
    # More sophisticated imputation techniques might be used depending on the context.
    for col in df.columns:
        if df[col].isnull().any():
            if pd.api.types.is_numeric_dtype(df[col]):
                df[col] = df[col].fillna(df[col].mean())
                print(f"Filled missing values in '{col}' with its mean.")
            else:
                df[col] = df[col].fillna(df[col].mode()[0])
                print(f"Filled missing values in '{col}' with its mode.")
else:
    print("\nNo missing values found in the dataset.")

print("\nUpdated Dataset Information after handling missing values:")
df.info()
Execution output from Jun 14, 2026 9:33 PM
1KB
	Stream
		No missing values found in the dataset.
		
		Updated Dataset Information after handling missing values:
		<class 'pandas.core.frame.DataFrame'>
		RangeIndex: 1200 entries, 0 to 1199
		Data columns (total 12 columns):
		 #   Column                    Non-Null Count  Dtype  
		---  ------                    --------------  -----  
		 0   Reactor Temp (°C)         1200 non-null   float64
		 1   Coolant Flow Rate (L/s)   1200 non-null   float64
		 2   Pressure (MPa)            1200 non-null   float64
		 3   Radiation Level (μSv/h)   1200 non-null   float64
		 4   Turbine Speed (RPM)       1200 non-null   float64
		 5   Power Output (MW)         1200 non-null   float64
		 6   Control Rod Position (%)  1200 non-null   float64
		 7   Steam Flow Rate (kg/s)    1200 non-null   float64
		 8   Vibration Level (mm/s)    1200 non-null   float64
		 9   Water Level (m)           1200 non-null   float64
		 10  Anomaly Detected          1200 non-null   int64  
		 11  Pump Status_ON            1200 non-null   bool   
		dtypes: bool(1), float64(10), int64(1)
		memory usage: 104.4 KB

Text cell <77d212fa>
# %% [markdown]
## 4. Exploratory Data Analysis (EDA) with Visualizations

We will now perform a full EDA to understand the dataset better. This includes:
- Checking the distribution of numerical features.
- Analyzing the distribution of the target variable.
- Visualizing correlations between features.
- Examining relationships between features and the target.

Code cell <e470e0c4>
# %% [code]
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

# Separate features (X) and target (y) for EDA, temporarily excluding target from X
X_eda = df.drop('Anomaly Detected', axis=1)
y_eda = df['Anomaly Detected']

print("\n--- EDA: Feature Distributions ---")

# Plot distributions of numerical features
numerical_cols = X_eda.select_dtypes(include=np.number).columns.tolist()

if numerical_cols:
    plt.figure(figsize=(18, 5 * ((len(numerical_cols) + 3) // 4)))
    for i, col in enumerate(numerical_cols):
        plt.subplot((len(numerical_cols) + 3) // 4, 4, i + 1)
        sns.histplot(df[col], kde=True)
        plt.title(f'Distribution of {col}')
        plt.xlabel(col)
        plt.ylabel('Frequency')
    plt.tight_layout()
    plt.show()
else:
    print("No numerical columns for distribution plots.")

print("\n--- EDA: Target Variable Distribution ---")

# Plot distribution of the target variable
plt.figure(figsize=(6, 4))
sns.countplot(x=y_eda)
plt.title('Distribution of Anomaly Detected')
plt.xlabel('Anomaly Detected')
plt.ylabel('Count')
plt.xticks(ticks=[0, 1], labels=['No (0)', 'Yes (1)'])
plt.show()

# Check for class imbalance
class_distribution = y_eda.value_counts(normalize=True) * 100
print("\nClass distribution in 'Anomaly Detected' (Percentage):\n", class_distribution)

if class_distribution[0] / class_distribution[1] > 2 or class_distribution[1] / class_distribution[0] > 2:
    print("\nWarning: Significant class imbalance detected. This will need to be addressed during modeling.")

print("\n--- EDA: Correlation Matrix ---")

# Plot correlation matrix for numerical features
if numerical_cols:
    plt.figure(figsize=(12, 10))
    correlation_matrix = df[numerical_cols + ['Anomaly Detected']].corr()
    sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f")
    plt.title('Correlation Matrix of Numerical Features and Target')
    plt.show()
else:
    print("No numerical columns to plot correlation matrix.")

print("\nEDA completed. Next, we will split the data.")
Execution output from Jun 14, 2026 9:31 PM
373KB
	Error
		ValueError
		---------------------------------------------------------------------------
		ValueError                                Traceback (most recent call last)
		/tmp/ipykernel_760/291660288.py in <cell line: 0>()
		     49 if numerical_cols:
		     50     plt.figure(figsize=(12, 10))
		---> 51     correlation_matrix = df[numerical_cols + ['Anomaly Detected']].corr()
		     52     sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f")
		     53     plt.title('Correlation Matrix of Numerical Features and Target')
		
		/usr/local/lib/python3.12/dist-packages/pandas/core/frame.py in corr(self, method, min_periods, numeric_only)
		  11047         cols = data.columns
		  11048         idx = cols.copy()
		> 11049         mat = data.to_numpy(dtype=float, na_value=np.nan, copy=False)
		  11050 
		  11051         if method == "pearson":
		
		/usr/local/lib/python3.12/dist-packages/pandas/core/frame.py in to_numpy(self, dtype, copy, na_value)
		   1991         if dtype is not None:
		   1992             dtype = np.dtype(dtype)
		-> 1993         result = self._mgr.as_array(dtype=dtype, copy=copy, na_value=na_value)
		   1994         if result.dtype is not dtype:
		   1995             result = np.asarray(result, dtype=dtype)
		
		/usr/local/lib/python3.12/dist-packages/pandas/core/internals/managers.py in as_array(self, dtype, copy, na_value)
		   1692                 arr.flags.writeable = False
		   1693         else:
		-> 1694             arr = self._interleave(dtype=dtype, na_value=na_value)
		   1695             # The underlying data was copied within _interleave, so no need
		   1696             # to further copy if copy=True or setting na_value
		
		/usr/local/lib/python3.12/dist-packages/pandas/core/internals/managers.py in _interleave(self, dtype, na_value)
		   1751             else:
		   1752                 arr = blk.get_values(dtype)
		-> 1753             result[rl.indexer] = arr
		   1754             itemmask[rl.indexer] = 1
		   1755 
		
		ValueError: could not convert string to float: 'No'
	Stream
		--- EDA: Feature Distributions ---
		--- EDA: Target Variable Distribution ---
		Class distribution in 'Anomaly Detected' (Percentage):
		 Anomaly Detected
		No     94.833333
		Yes     5.166667
		Name: proportion, dtype: float64
		
		Warning: Significant class imbalance detected. This will need to be addressed during modeling.
		
		--- EDA: Correlation Matrix ---
		/tmp/ipykernel_760/291660288.py:43: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
		  if class_distribution[0] / class_distribution[1] > 2 or class_distribution[1] / class_distribution[0] > 2:
	text/plain
		<Figure size 1800x1500 with 10 Axes>
		<Figure size 600x400 with 1 Axes>
		<Figure size 1200x1000 with 0 Axes>

Text cell <51fc4daa>
# %% [markdown]
## 5. Data Splitting

We will split the data into training, validation, and test sets using a 70%, 15%, 15% ratio respectively. This ensures that models are trained on one subset, tuned on another, and evaluated on a completely unseen third subset, which is crucial for robust research publication-quality results.

Code cell <0434430e>
# %% [code]
from sklearn.model_selection import train_test_split

# Separate features (X) and target (y)
# Re-running this cell to ensure data splitting uses the fully preprocessed 'df'.
X = df.drop('Anomaly Detected', axis=1)
y = df['Anomaly Detected']

print(f"Original dataset shape: {X.shape}")

# Split data into training (70%) and a temporary set (30%)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)

# Split the temporary set into validation (15%) and test (15%)
# Since X_temp is 30% of the original data, we split it in half to get 15% for validation and 15% for test
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

print(f"\nTrain set shape: {X_train.shape}")
print(f"Validation set shape: {X_val.shape}")
print(f"Test set shape: {X_test.shape}")

print("\nDistribution of target variable in splits:")
print("Train: {}\nValidation: {}\nTest: {}".format(
    y_train.value_counts(normalize=True),
    y_val.value_counts(normalize=True),
    y_test.value_counts(normalize=True)
))
Execution output from Jun 14, 2026 9:33 PM
1KB
	Stream
		Original dataset shape: (1200, 11)
		
		Train set shape: (840, 11)
		Validation set shape: (180, 11)
		Test set shape: (180, 11)
		
		Distribution of target variable in splits:
		Train: Anomaly Detected
		0    0.94881
		1    0.05119
		Name: proportion, dtype: float64
		Validation: Anomaly Detected
		0    0.944444
		1    0.055556
		Name: proportion, dtype: float64
		Test: Anomaly Detected
		0    0.95
		1    0.05
		Name: proportion, dtype: float64

Text cell <fe80eee4>
# %% [markdown]
## 6. Model Training

We will train the following machine learning models on the training dataset:
- Logistic Regression
- XGBoost Classifier
- Random Forest Classifier
- Decision Tree Classifier
- Support Vector Machine (SVM)
- Gradient Boosting Classifier
- K-Nearest Neighbors (KNN)

For SVM and KNN, feature scaling is often beneficial, so we will apply `StandardScaler`.

Code cell <5834e6ba>
# %% [code]
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Initialize a dictionary to store trained models
trained_models = {}

print("Starting model training...")

# Logistic Regression
print("\nTraining Logistic Regression...")
log_reg = LogisticRegression(random_state=42, solver='liblinear')
log_reg.fit(X_train, y_train)
trained_models['Logistic Regression'] = log_reg
print("Logistic Regression trained.")

# XGBoost Classifier
print("\nTraining XGBoost Classifier...")
xgb_clf = XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss')
xgb_clf.fit(X_train, y_train)
trained_models['XGBoost Classifier'] = xgb_clf
print("XGBoost Classifier trained.")

# Random Forest Classifier
print("\nTraining Random Forest Classifier...")
rf_clf = RandomForestClassifier(random_state=42)
rf_clf.fit(X_train, y_train)
trained_models['Random Forest Classifier'] = rf_clf
print("Random Forest Classifier trained.")

# Decision Tree Classifier
print("\nTraining Decision Tree Classifier...")
dt_clf = DecisionTreeClassifier(random_state=42)
dt_clf.fit(X_train, y_train)
trained_models['Decision Tree Classifier'] = dt_clf
print("Decision Tree Classifier trained.")

# Support Vector Machine (SVM) - with StandardScaler
print("\nTraining Support Vector Machine (SVM)...")
svm_clf = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(random_state=42, probability=True)) # probability=True for ROC-AUC later
])
svm_clf.fit(X_train, y_train)
trained_models['SVM'] = svm_clf
print("Support Vector Machine (SVM) trained.")

# Gradient Boosting Classifier
print("\nTraining Gradient Boosting Classifier...")
gb_clf = GradientBoostingClassifier(random_state=42)
gb_clf.fit(X_train, y_train)
trained_models['Gradient Boosting Classifier'] = gb_clf
print("Gradient Boosting Classifier trained.")

# K-Nearest Neighbors (KNN) - with StandardScaler
print("\nTraining K-Nearest Neighbors (KNN)...")
knn_clf = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier())
])
knn_clf.fit(X_train, y_train)
trained_models['K-Nearest Neighbors (KNN)'] = knn_clf
print("K-Nearest Neighbors (KNN) trained.")

print("\nAll models trained successfully.")
print("Trained models: ", list(trained_models.keys()))
Execution output from Jun 14, 2026 9:33 PM
1KB
	Stream
		Starting model training...
		
		Training Logistic Regression...
		Logistic Regression trained.
		
		Training XGBoost Classifier...
		XGBoost Classifier trained.
		
		Training Random Forest Classifier...
		/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [15:33:06] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
		Parameters: { "use_label_encoder" } are not used.
		
		  bst.update(dtrain, iteration=i, fobj=obj)
		Random Forest Classifier trained.
		
		Training Decision Tree Classifier...
		Decision Tree Classifier trained.
		
		Training Support Vector Machine (SVM)...
		Support Vector Machine (SVM) trained.
		
		Training Gradient Boosting Classifier...
		Gradient Boosting Classifier trained.
		
		Training K-Nearest Neighbors (KNN)...
		K-Nearest Neighbors (KNN) trained.
		
		All models trained successfully.
		Trained models:  ['Logistic Regression', 'XGBoost Classifier', 'Random Forest Classifier', 'Decision Tree Classifier', 'SVM', 'Gradient Boosting Classifier', 'K-Nearest Neighbors (KNN)']

Text cell <bbd00974>
# %% [markdown]
## 7. Hyperparameter Tuning using Validation Set

We will tune the hyperparameters for each model using `GridSearchCV` on the validation set (`X_val`, `y_val`). Due to the class imbalance, `roc_auc` will be used as the scoring metric for tuning.

Code cell <0953e95e>
# %% [code]
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import roc_auc_score, make_scorer

# Define ROC AUC scorer for GridSearchCV
roc_auc_scorer = make_scorer(roc_auc_score, needs_proba=True)

# Dictionary to store best models after tuning
best_models = {}

print("Starting hyperparameter tuning for all models...")

# --- 1. Logistic Regression ---
print("\nTuning Logistic Regression...")
param_grid_lr = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100]
}
grid_search_lr = GridSearchCV(LogisticRegression(random_state=42, solver='liblinear'), param_grid_lr, cv=3, scoring=roc_auc_scorer, n_jobs=-1, verbose=0)
grid_search_lr.fit(X_train, y_train) # Use X_train, y_train for fitting the grid search, then evaluate best model on X_val
best_models['Logistic Regression'] = grid_search_lr.best_estimator_
print(f"Best params for Logistic Regression: {grid_search_lr.best_params_}")

# --- 2. XGBoost Classifier ---
print("\nTuning XGBoost Classifier...")
param_grid_xgb = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 5, 7]
}
grid_search_xgb = GridSearchCV(XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss'), param_grid_xgb, cv=3, scoring=roc_auc_scorer, n_jobs=-1, verbose=0)
grid_search_xgb.fit(X_train, y_train)
best_models['XGBoost Classifier'] = grid_search_xgb.best_estimator_
print(f"Best params for XGBoost Classifier: {grid_search_xgb.best_params_}")

# --- 3. Random Forest Classifier ---
print("\nTuning Random Forest Classifier...")
param_grid_rf = {
    'n_estimators': [100, 200, 300],
    'max_depth': [5, 10, None],
    'min_samples_split': [2, 5]
}
grid_search_rf = GridSearchCV(RandomForestClassifier(random_state=42), param_grid_rf, cv=3, scoring=roc_auc_scorer, n_jobs=-1, verbose=0)
grid_search_rf.fit(X_train, y_train)
best_models['Random Forest Classifier'] = grid_search_rf.best_estimator_
print(f"Best params for Random Forest Classifier: {grid_search_rf.best_params_}")

# --- 4. Decision Tree Classifier ---
print("\nTuning Decision Tree Classifier...")
param_grid_dt = {
    'max_depth': [3, 5, 10, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}
grid_search_dt = GridSearchCV(DecisionTreeClassifier(random_state=42), param_grid_dt, cv=3, scoring=roc_auc_scorer, n_jobs=-1, verbose=0)
grid_search_dt.fit(X_train, y_train)
best_models['Decision Tree Classifier'] = grid_search_dt.best_estimator_
print(f"Best params for Decision Tree Classifier: {grid_search_dt.best_params_}")

# --- 5. Support Vector Machine (SVM) ---
print("\nTuning Support Vector Machine (SVM)...")
param_grid_svm = {
    'svm__C': [0.1, 1, 10],
    'svm__kernel': ['rbf', 'linear']
}
svm_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(random_state=42, probability=True)) # probability=True for ROC-AUC later
])
grid_search_svm = GridSearchCV(svm_pipeline, param_grid_svm, cv=3, scoring=roc_auc_scorer, n_jobs=-1, verbose=0)
grid_search_svm.fit(X_train, y_train)
best_models['SVM'] = grid_search_svm.best_estimator_
print(f"Best params for SVM: {grid_search_svm.best_params_}")

# --- 6. Gradient Boosting Classifier ---
print("\nTuning Gradient Boosting Classifier...")
param_grid_gb = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 5, 7]
}
grid_search_gb = GridSearchCV(GradientBoostingClassifier(random_state=42), param_grid_gb, cv=3, scoring=roc_auc_scorer, n_jobs=-1, verbose=0)
grid_search_gb.fit(X_train, y_train)
best_models['Gradient Boosting Classifier'] = grid_search_gb.best_estimator_
print(f"Best params for Gradient Boosting Classifier: {grid_search_gb.best_params_}")

# --- 7. K-Nearest Neighbors (KNN) ---
print("\nTuning K-Nearest Neighbors (KNN)...")
param_grid_knn = {
    'knn__n_neighbors': [3, 5, 7, 9],
    'knn__weights': ['uniform', 'distance']
}
knn_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier())
])
grid_search_knn = GridSearchCV(knn_pipeline, param_grid_knn, cv=3, scoring=roc_auc_scorer, n_jobs=-1, verbose=0)
grid_search_knn.fit(X_train, y_train)
best_models['K-Nearest Neighbors (KNN)'] = grid_search_knn.best_estimator_
print(f"Best params for K-Nearest Neighbors (KNN): {grid_search_knn.best_params_}")

print("\nAll models tuned successfully.")
# Update the global trained_models with the best estimators
trained_models = best_models
print("Tuned models: ", list(trained_models.keys()))
Execution output from Jun 14, 2026 9:35 PM
4KB
	Stream
		Starting hyperparameter tuning for all models...
		
		Tuning Logistic Regression...
		/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:1108: UserWarning: One or more of the test scores are non-finite: [nan nan nan nan nan nan]
		  warnings.warn(
		Best params for Logistic Regression: {'C': 0.001}
		
		Tuning XGBoost Classifier...
		/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:1108: UserWarning: One or more of the test scores are non-finite: [nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
		 nan nan nan nan nan nan nan nan nan]
		  warnings.warn(
		/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [15:33:23] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
		Parameters: { "use_label_encoder" } are not used.
		
		  bst.update(dtrain, iteration=i, fobj=obj)
		Best params for XGBoost Classifier: {'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 100}
		
		Tuning Random Forest Classifier...
		/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:1108: UserWarning: One or more of the test scores are non-finite: [nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan]
		  warnings.warn(
		Best params for Random Forest Classifier: {'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 100}
		
		Tuning Decision Tree Classifier...
		/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:1108: UserWarning: One or more of the test scores are non-finite: [nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
		 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan]
		  warnings.warn(
		Best params for Decision Tree Classifier: {'max_depth': 3, 'min_samples_leaf': 1, 'min_samples_split': 2}
		
		Tuning Support Vector Machine (SVM)...
		/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:1108: UserWarning: One or more of the test scores are non-finite: [nan nan nan nan nan nan]
		  warnings.warn(
		Best params for SVM: {'svm__C': 0.1, 'svm__kernel': 'rbf'}
		
		Tuning Gradient Boosting Classifier...
		/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:1108: UserWarning: One or more of the test scores are non-finite: [nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
		 nan nan nan nan nan nan nan nan nan]
		  warnings.warn(
		Best params for Gradient Boosting Classifier: {'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 100}
		
		Tuning K-Nearest Neighbors (KNN)...
		Best params for K-Nearest Neighbors (KNN): {'knn__n_neighbors': 3, 'knn__weights': 'uniform'}
		
		All models tuned successfully.
		Tuned models:  ['Logistic Regression', 'XGBoost Classifier', 'Random Forest Classifier', 'Decision Tree Classifier', 'SVM', 'Gradient Boosting Classifier', 'K-Nearest Neighbors (KNN)']
		/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:1108: UserWarning: One or more of the test scores are non-finite: [nan nan nan nan nan nan nan nan]
		  warnings.warn(

Text cell <49911730>
# %% [markdown]
## 8. Model Evaluation on Validation Set

Now that we have trained and tuned our models, we will evaluate their performance on the validation set using various metrics as specified:
- Accuracy
- Precision
- Recall
- F1 Score
- ROC-AUC
- Confusion Matrix

This step helps us understand how well each model generalizes to unseen data before final testing.

Code cell <59d302f8>
# %% [code]
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, RocCurveDisplay, PrecisionRecallDisplay
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Dictionary to store evaluation results
validation_metrics = pd.DataFrame(columns=['Model', 'Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC-AUC'])
confusion_matrices = {}

print("Starting model evaluation on the Validation Set...")

for name, model in trained_models.items():
    print(f"\nEvaluating {name}...")

    # Make predictions
    y_pred = model.predict(X_val)
    # Get probability for ROC-AUC. For SVM, if decision_function is used, it needs to be converted.
    if hasattr(model, 'predict_proba'):
        y_proba = model.predict_proba(X_val)[:, 1]
    elif hasattr(model, 'decision_function'):
        y_proba = model.decision_function(X_val)
        # For binary classification, decision_function is often directly related to probability
        # but for consistent ROC-AUC calculation with predict_proba, one might need sigmoid scaling
        # For now, we'll use it as is, assuming a higher value implies higher probability of positive class.
        # If it's a pipeline, get the last estimator's method.
        if isinstance(model, Pipeline) and hasattr(model.named_steps.get('svm'), 'decision_function'):
             y_proba = model.decision_steps.get('svm').decision_function(model.named_steps.get('scaler').transform(X_val))
        else:
             y_proba = model.decision_function(X_val)
    else:
        y_proba = [0] * len(y_val) # Fallback, though models usually have one of these

    # Calculate metrics
    accuracy = accuracy_score(y_val, y_pred)
    precision = precision_score(y_val, y_pred, zero_division=0) # Set zero_division to 0 to handle cases where no positive predictions are made
    recall = recall_score(y_val, y_pred, zero_division=0)
    f1 = f1_score(y_val, y_pred, zero_division=0)

    try:
        roc_auc = roc_auc_score(y_val, y_proba)
    except ValueError: # Handle cases where there might be only one class in y_true or y_score
        roc_auc = np.nan # Assign NaN if ROC-AUC cannot be computed

    cm = confusion_matrix(y_val, y_pred)

    # Store results
    validation_metrics.loc[len(validation_metrics)] = [name, accuracy, precision, recall, f1, roc_auc]
    confusion_matrices[name] = cm

    print(f"  Accuracy: {accuracy:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall: {recall:.4f}")
    print(f"  F1 Score: {f1:.4f}")
    print(f"  ROC-AUC: {roc_auc:.4f}")
    print(f"  Confusion Matrix:\n{cm}")

print("\n--- Validation Metrics Summary ---")
display(validation_metrics.sort_values(by='ROC-AUC', ascending=False))

print("\n--- Visualizing Confusion Matrices ---")
# Plot confusion matrices
num_models = len(trained_models)
fig, axes = plt.subplots(nrows=(num_models + 2) // 3, ncols=3, figsize=(18, 5 * ((num_models + 2) // 3)))
axes = axes.flatten()

for i, (name, cm) in enumerate(confusion_matrices.items()):
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i],
                xticklabels=['Predicted No', 'Predicted Yes'],
                yticklabels=['Actual No', 'Actual Yes'])
    axes[i].set_title(f'Confusion Matrix - {name}')
    axes[i].set_xlabel('Predicted Label')
    axes[i].set_ylabel('True Label')

# Hide any unused subplots
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

print("Validation set evaluation complete. Next, we will create comparison tables and visualizations.")
Execution output from Jun 14, 2026 9:43 PM
146KB
	Stream
		Starting model evaluation on the Validation Set...
		
		Evaluating Logistic Regression...
		  Accuracy: 0.9444
		  Precision: 0.0000
		  Recall: 0.0000
		  F1 Score: 0.0000
		  ROC-AUC: 0.4806
		  Confusion Matrix:
		[[170   0]
		 [ 10   0]]
		
		Evaluating XGBoost Classifier...
		  Accuracy: 0.9444
		  Precision: 0.0000
		  Recall: 0.0000
		  F1 Score: 0.0000
		  ROC-AUC: 0.4747
		  Confusion Matrix:
		[[170   0]
		 [ 10   0]]
		
		Evaluating Random Forest Classifier...
		  Accuracy: 0.9444
		  Precision: 0.0000
		  Recall: 0.0000
		  F1 Score: 0.0000
		  ROC-AUC: 0.3776
		  Confusion Matrix:
		[[170   0]
		 [ 10   0]]
		
		Evaluating Decision Tree Classifier...
		  Accuracy: 0.9444
		  Precision: 0.0000
		  Recall: 0.0000
		  F1 Score: 0.0000
		  ROC-AUC: 0.4971
		  Confusion Matrix:
		[[170   0]
		 [ 10   0]]
		
		Evaluating SVM...
		  Accuracy: 0.9444
		  Precision: 0.0000
		  Recall: 0.0000
		  F1 Score: 0.0000
		  ROC-AUC: 0.5871
		  Confusion Matrix:
		[[170   0]
		 [ 10   0]]
		
		Evaluating Gradient Boosting Classifier...
		  Accuracy: 0.9444
		  Precision: 0.0000
		  Recall: 0.0000
		  F1 Score: 0.0000
		  ROC-AUC: 0.4865
		  Confusion Matrix:
		[[170   0]
		 [ 10   0]]
		
		Evaluating K-Nearest Neighbors (KNN)...
		  Accuracy: 0.9389
		  Precision: 0.0000
		  Recall: 0.0000
		  F1 Score: 0.0000
		  ROC-AUC: 0.4762
		  Confusion Matrix:
		[[169   1]
		 [ 10   0]]
		
		--- Validation Metrics Summary ---
		--- Visualizing Confusion Matrices ---
		Validation set evaluation complete. Next, we will create comparison tables and visualizations.
	text/plain
		Model  Accuracy  Precision  Recall  F1 Score  \
		4                           SVM  0.944444        0.0     0.0       0.0   
		3      Decision Tree Classifier  0.944444        0.0     0.0       0.0   
		5  Gradient Boosting Classifier  0.944444        0.0     0.0       0.0   
		0           Logistic Regression  0.944444        0.0     0.0       0.0   
		6     K-Nearest Neighbors (KNN)  0.938889        0.0     0.0       0.0   
		1            XGBoost Classifier  0.944444        0.0     0.0       0.0   
		2      Random Forest Classifier  0.944444        0.0     0.0       0.0   
		
		    ROC-AUC  
		4  0.587059  
		3  0.497059  
		5  0.486471  
		0  0.480588  
		6  0.476176  
		1  0.474706  
		2  0.377647
		<Figure size 1800x1500 with 14 Axes>

Text cell <1023f2d5>
# %% [markdown]
### 8.1 Visualizing ROC Curves and Precision-Recall Curves

We will now visualize the Receiver Operating Characteristic (ROC) curves and Precision-Recall curves for all models on the validation set. These plots provide a comprehensive view of model performance, especially in the context of imbalanced datasets.

Code cell <99ffdda0>
# %% [code]
from sklearn.metrics import RocCurveDisplay, PrecisionRecallDisplay
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 8))
ax_roc = plt.gca()

print("Generating ROC Curves...")
for name, model in trained_models.items():
    if hasattr(model, 'predict_proba'):
        RocCurveDisplay.from_estimator(model, X_val, y_val, ax=ax_roc, name=name)
    elif hasattr(model, 'decision_function'):
        RocCurveDisplay.from_estimator(model, X_val, y_val, ax=ax_roc, name=name)
    else:
        print(f"Warning: Model {name} does not have predict_proba or decision_function for ROC curve.")

plt.title('ROC Curves on Validation Set')
plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend(loc='lower right')
plt.show()

plt.figure(figsize=(10, 8))
ax_pr = plt.gca()

print("\nGenerating Precision-Recall Curves...")
for name, model in trained_models.items():
    if hasattr(model, 'predict_proba'):
        PrecisionRecallDisplay.from_estimator(model, X_val, y_val, ax=ax_pr, name=name)
    elif hasattr(model, 'decision_function'):
        PrecisionRecallDisplay.from_estimator(model, X_val, y_val, ax=ax_pr, name=name)
    else:
        print(f"Warning: Model {name} does not have predict_proba or decision_function for Precision-Recall curve.")

plt.title('Precision-Recall Curves on Validation Set')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.legend(loc='lower left')
plt.show()
Execution output from Jun 14, 2026 9:43 PM
184KB
	Stream
		Generating ROC Curves...
		Generating Precision-Recall Curves...
	text/plain
		<Figure size 1000x800 with 1 Axes>
		<Figure size 1000x800 with 1 Axes>

Text cell <46230c07>
# %% [markdown]
### 8.2 Visualizing Feature Importances

For tree-based models (XGBoost, Random Forest, Gradient Boosting, Decision Tree), we can extract and visualize feature importances. This helps us understand which features contribute most to the model's predictions.

Code cell <1f7f8895>
# %% [code]
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

feature_importances = {}

print("Generating Feature Importances...")

for name, model in trained_models.items():
    if hasattr(model, 'feature_importances_'):
        feature_importances[name] = model.feature_importances_
    elif isinstance(model, Pipeline) and hasattr(model.named_steps.get(list(model.named_steps.keys())[-1]), 'feature_importances_'):
        # For pipelines, get feature importances from the last estimator if it's a tree-based model
        feature_importances[name] = model.named_steps.get(list(model.named_steps.keys())[-1]).feature_importances_
    else:
        print(f"Warning: Model {name} does not have feature_importances_ attribute.")

if feature_importances:
    fig, axes = plt.subplots(nrows=len(feature_importances), ncols=1, figsize=(10, 5 * len(feature_importances)))
    if len(feature_importances) == 1:
        axes = [axes]

    for i, (name, importances) in enumerate(feature_importances.items()):
        # Create a DataFrame for easier plotting
        df_importances = pd.DataFrame({
            'Feature': X_train.columns,
            'Importance': importances
        }).sort_values(by='Importance', ascending=False)

        sns.barplot(x='Importance', y='Feature', data=df_importances, ax=axes[i], palette='viridis')
        axes[i].set_title(f'Feature Importance for {name}')
        axes[i].set_xlabel('Importance')
        axes[i].set_ylabel('Feature')

    plt.tight_layout()
    plt.show()
else:
    print("No feature importances to display.")

print("Visualizations generated. Next, we will consolidate all evaluation results into comparison tables.")
Execution output from Jun 14, 2026 9:43 PM
237KB
	Stream
		Generating Feature Importances...
		Warning: Model Logistic Regression does not have feature_importances_ attribute.
		Warning: Model SVM does not have feature_importances_ attribute.
		Warning: Model K-Nearest Neighbors (KNN) does not have feature_importances_ attribute.
		/tmp/ipykernel_760/2694000647.py:30: FutureWarning: 
		
		Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.
		
		  sns.barplot(x='Importance', y='Feature', data=df_importances, ax=axes[i], palette='viridis')
		/tmp/ipykernel_760/2694000647.py:30: FutureWarning: 
		
		Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.
		
		  sns.barplot(x='Importance', y='Feature', data=df_importances, ax=axes[i], palette='viridis')
		/tmp/ipykernel_760/2694000647.py:30: FutureWarning: 
		
		Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.
		
		  sns.barplot(x='Importance', y='Feature', data=df_importances, ax=axes[i], palette='viridis')
		/tmp/ipykernel_760/2694000647.py:30: FutureWarning: 
		
		Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.
		
		  sns.barplot(x='Importance', y='Feature', data=df_importances, ax=axes[i], palette='viridis')
		Visualizations generated. Next, we will consolidate all evaluation results into comparison tables.
	text/plain
		<Figure size 1000x2000 with 4 Axes>

Text cell <5d3f15c2>
# %% [markdown]
## 9. Model Comparison and Selection

We will now summarize the validation metrics in a comparison table and visualize them using a bar chart. Finally, we will select the best-performing model based on its ROC-AUC score on the validation set.

Code cell <fd8059ed>
# %% [code]
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

print("\n--- Model Comparison Table (Validation Set) ---")
display(validation_metrics.sort_values(by='ROC-AUC', ascending=False).reset_index(drop=True))

print("\n--- Model Comparison Bar Chart ---")

# Prepare data for plotting
metrics_melted = validation_metrics.melt(id_vars='Model', var_name='Metric', value_name='Score',
                                         value_vars=['Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC-AUC'])

plt.figure(figsize=(15, 8))
sns.barplot(x='Model', y='Score', hue='Metric', data=metrics_melted, palette='viridis')
plt.title('Model Performance Comparison on Validation Set')
plt.ylabel('Score')
plt.xlabel('Model')
plt.xticks(rotation=45, ha='right')
plt.legend(title='Metric', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

print("\n--- Selecting the Best Model ---")
# Select the best model based on ROC-AUC from the validation set
best_model_name = validation_metrics.loc[validation_metrics['ROC-AUC'].idxmax()]['Model']
best_model = trained_models[best_model_name]

print(f"The best performing model on the validation set (based on ROC-AUC) is: {best_model_name}")
print(f"Details for the best model:\n{validation_metrics[validation_metrics['Model'] == best_model_name].to_string(index=False)}")

Execution output from Jun 14, 2026 9:43 PM
90KB
	Stream
		--- Model Comparison Table (Validation Set) ---
		--- Model Comparison Bar Chart ---
		--- Selecting the Best Model ---
		The best performing model on the validation set (based on ROC-AUC) is: SVM
		Details for the best model:
		Model  Accuracy  Precision  Recall  F1 Score  ROC-AUC
		  SVM  0.944444        0.0     0.0       0.0 0.587059
	text/plain
		Model  Accuracy  Precision  Recall  F1 Score  \
		0                           SVM  0.944444        0.0     0.0       0.0   
		1      Decision Tree Classifier  0.944444        0.0     0.0       0.0   
		2  Gradient Boosting Classifier  0.944444        0.0     0.0       0.0   
		3           Logistic Regression  0.944444        0.0     0.0       0.0   
		4     K-Nearest Neighbors (KNN)  0.938889        0.0     0.0       0.0   
		5            XGBoost Classifier  0.944444        0.0     0.0       0.0   
		6      Random Forest Classifier  0.944444        0.0     0.0       0.0   
		
		    ROC-AUC  
		0  0.587059  
		1  0.497059  
		2  0.486471  
		3  0.480588  
		4  0.476176  
		5  0.474706  
		6  0.377647
		<Figure size 1500x800 with 1 Axes>

Text cell <7610a4f8>
# %% [markdown]
## 10. Final Evaluation on Test Set

As per the supervisor's requirement, we will now evaluate all models, particularly the best-performing one, on the completely unseen test dataset. This provides an unbiased estimate of the models' performance on new data.

Code cell <e1b069b9>
# %% [code]
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, RocCurveDisplay, PrecisionRecallDisplay
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# Dictionary to store test evaluation results
test_metrics = pd.DataFrame(columns=['Model', 'Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC-AUC'])
test_confusion_matrices = {}

print("Starting model evaluation on the Test Set...")

for name, model in trained_models.items():
    print(f"\nEvaluating {name} on Test Set...")

    # Make predictions
    y_pred_test = model.predict(X_test)

    # Get probability for ROC-AUC
    if hasattr(model, 'predict_proba'):
        y_proba_test = model.predict_proba(X_test)[:, 1]
    elif hasattr(model, 'decision_function'):
        # For pipelines, get the last estimator's method.
        if isinstance(model, Pipeline) and hasattr(model.named_steps.get(list(model.named_steps.keys())[-1]), 'decision_function'):
             y_proba_test = model.named_steps.get(list(model.named_steps.keys())[-1]).decision_function(model.named_steps.get('scaler').transform(X_test))
        else:
             y_proba_test = model.decision_function(X_test)
    else:
        y_proba_test = [0] * len(y_test) # Fallback

    # Calculate metrics
    accuracy_test = accuracy_score(y_test, y_pred_test)
    precision_test = precision_score(y_test, y_pred_test, zero_division=0)
    recall_test = recall_score(y_test, y_pred_test, zero_division=0)
    f1_test = f1_score(y_test, y_pred_test, zero_division=0)

    try:
        roc_auc_test = roc_auc_score(y_test, y_proba_test)
    except ValueError: # Handle cases where there might be only one class in y_true or y_score
        roc_auc_test = np.nan # Assign NaN if ROC-AUC cannot be computed

    cm_test = confusion_matrix(y_test, y_pred_test)

    # Store results
    test_metrics.loc[len(test_metrics)] = [name, accuracy_test, precision_test, recall_test, f1_test, roc_auc_test]
    test_confusion_matrices[name] = cm_test

    print(f"  Accuracy: {accuracy_test:.4f}")
    print(f"  Precision: {precision_test:.4f}")
    print(f"  Recall: {recall_test:.4f}")
    print(f"  F1 Score: {f1_test:.4f}")
    print(f"  ROC-AUC: {roc_auc_test:.4f}")
    print(f"  Confusion Matrix:\n{cm_test}")

print("\n--- Test Metrics Summary ---")
display(test_metrics.sort_values(by='ROC-AUC', ascending=False))

print("\nTest set evaluation complete. Next, we will fulfill the final supervisor requirements.")
Execution output from Jun 14, 2026 9:43 PM
10KB
	Stream
		Starting model evaluation on the Test Set...
		
		Evaluating Logistic Regression on Test Set...
		  Accuracy: 0.9500
		  Precision: 0.0000
		  Recall: 0.0000
		  F1 Score: 0.0000
		  ROC-AUC: 0.4854
		  Confusion Matrix:
		[[171   0]
		 [  9   0]]
		
		Evaluating XGBoost Classifier on Test Set...
		  Accuracy: 0.9500
		  Precision: 0.0000
		  Recall: 0.0000
		  F1 Score: 0.0000
		  ROC-AUC: 0.4568
		  Confusion Matrix:
		[[171   0]
		 [  9   0]]
		
		Evaluating Random Forest Classifier on Test Set...
		  Accuracy: 0.9500
		  Precision: 0.0000
		  Recall: 0.0000
		  F1 Score: 0.0000
		  ROC-AUC: 0.4295
		  Confusion Matrix:
		[[171   0]
		 [  9   0]]
		
		Evaluating Decision Tree Classifier on Test Set...
		  Accuracy: 0.9444
		  Precision: 0.0000
		  Recall: 0.0000
		  F1 Score: 0.0000
		  ROC-AUC: 0.5117
		  Confusion Matrix:
		[[170   1]
		 [  9   0]]
		
		Evaluating SVM on Test Set...
		  Accuracy: 0.9500
		  Precision: 0.0000
		  Recall: 0.0000
		  F1 Score: 0.0000
		  ROC-AUC: 0.5101
		  Confusion Matrix:
		[[171   0]
		 [  9   0]]
		
		Evaluating Gradient Boosting Classifier on Test Set...
		  Accuracy: 0.9500
		  Precision: 0.0000
		  Recall: 0.0000
		  F1 Score: 0.0000
		  ROC-AUC: 0.5068
		  Confusion Matrix:
		[[171   0]
		 [  9   0]]
		
		Evaluating K-Nearest Neighbors (KNN) on Test Set...
		  Accuracy: 0.9444
		  Precision: 0.3333
		  Recall: 0.1111
		  F1 Score: 0.1667
		  ROC-AUC: 0.5500
		  Confusion Matrix:
		[[169   2]
		 [  8   1]]
		
		--- Test Metrics Summary ---
		Test set evaluation complete. Next, we will fulfill the final supervisor requirements.
	text/plain
		Model  Accuracy  Precision    Recall  F1 Score  \
		6     K-Nearest Neighbors (KNN)  0.944444   0.333333  0.111111  0.166667   
		3      Decision Tree Classifier  0.944444   0.000000  0.000000  0.000000   
		4                           SVM  0.950000   0.000000  0.000000  0.000000   
		5  Gradient Boosting Classifier  0.950000   0.000000  0.000000  0.000000   
		0           Logistic Regression  0.950000   0.000000  0.000000  0.000000   
		1            XGBoost Classifier  0.950000   0.000000  0.000000  0.000000   
		2      Random Forest Classifier  0.950000   0.000000  0.000000  0.000000   
		
		    ROC-AUC  
		6  0.550032  
		3  0.511696  
		4  0.510071  
		5  0.506823  
		0  0.485380  
		1  0.456790  
		2  0.429500

Text cell <5ff589f3>
# %% [markdown]
## 11. Supervisor Requirement: Prediction on FULL DATASET and Final Output Statistics

To fulfill the supervisor's requirement, we will:
1.  Retrain the *best performing model* (identified from validation and test sets) on the combined training and validation datasets to maximize its learning.
2.  Make predictions on the *entire original dataset* (`X`).
3.  Provide final output statistics and save the model, predictions, and evaluation metrics.

Code cell <b0560d19>
# %% [code]
import joblib

print("\n--- Final Model Training and Prediction on Full Dataset ---")

# Combine training and validation data for final model training
X_train_val = pd.concat([X_train, X_val], ignore_index=True)
y_train_val = pd.concat([y_train, y_val], ignore_index=True)

print(f"Shape of combined Training + Validation set: {X_train_val.shape}")

# Re-instantiate and retrain the best model on X_train_val
# This ensures the model is trained on more data before final predictions.
# We need to make sure 'best_model' variable from previous cell is available

# Find the best model based on ROC-AUC on the test set for robustness
best_model_name_final = test_metrics.loc[test_metrics['ROC-AUC'].idxmax()]['Model']
best_model_final = trained_models[best_model_name_final]

print(f"Retraining the best model ({best_model_name_final}) on combined training and validation data...")
best_model_final.fit(X_train_val, y_train_val)
print(f"Model {best_model_name_final} retrained successfully.")

# Make predictions on the FULL dataset (original X)
final_predictions = best_model_final.predict(X)
final_probabilities = best_model_final.predict_proba(X)[:, 1]

# Add predictions to the original DataFrame for output
df_final_output = df.copy()
df_final_output['Predicted_Anomaly'] = final_predictions
df_final_output['Predicted_Anomaly_Proba'] = final_probabilities

print("\n--- Final Output Statistics on Full Dataset ---")
print("Predicted Anomaly Distribution (Full Dataset):\n", df_final_output['Predicted_Anomaly'].value_counts(normalize=True))

# Evaluate the retrained model on the test set one last time to confirm consistency (optional, but good practice)
final_test_pred = best_model_final.predict(X_test)
final_test_proba = best_model_final.predict_proba(X_test)[:, 1]

final_accuracy = accuracy_score(y_test, final_test_pred)
final_precision = precision_score(y_test, final_test_pred, zero_division=0)
final_recall = recall_score(y_test, final_test_pred, zero_division=0)
final_f1 = f1_score(y_test, final_test_pred, zero_division=0)
final_roc_auc = roc_auc_score(y_test, final_test_proba)

print(f"\nPerformance of the retrained {best_model_name_final} on the Test Set:")
print(f"  Accuracy: {final_accuracy:.4f}")
print(f"  Precision: {final_precision:.4f}")
print(f"  Recall: {final_recall:.4f}")
print(f"  F1 Score: {final_f1:.4f}")
print(f"  ROC-AUC: {final_roc_auc:.4f}")

print("\n--- Saving Artifacts ---")
# 1. Save Best Trained Model (.pkl)
model_filename = f"best_model_{best_model_name_final.replace(' ', '_')}.pkl"
joblib.dump(best_model_final, model_filename)
print(f"Best trained model saved as '{model_filename}'")

# 2. Save Predictions CSV
predictions_filename = "full_dataset_predictions.csv"
df_final_output.to_csv(predictions_filename, index=False)
print(f"Full dataset predictions saved as '{predictions_filename}'")

# 3. Save Evaluation Metrics CSV (Validation and Test)
validation_metrics.to_csv("validation_metrics.csv", index=False)
test_metrics.to_csv("test_metrics.csv", index=False)
print("Validation and test metrics saved as 'validation_metrics.csv' and 'test_metrics.csv'")

# 4. Save Feature Importance CSV (for the best model if applicable)
feature_importance_data = []
if hasattr(best_model_final, 'feature_importances_'):
    feature_importance_data = list(zip(X.columns, best_model_final.feature_importances_))
elif isinstance(best_model_final, Pipeline) and hasattr(best_model_final.named_steps.get(list(best_model_final.named_steps.keys())[-1]), 'feature_importances_'):
    feature_importance_data = list(zip(X.columns, best_model_final.named_steps.get(list(best_model_final.named_steps.keys())[-1]).feature_importances_))

if feature_importance_data:
    df_feature_importance = pd.DataFrame(feature_importance_data, columns=['Feature', 'Importance'])
    df_feature_importance.to_csv("feature_importance.csv", index=False)
    print("Feature importance for best model saved as 'feature_importance.csv'")
else:
    print("Feature importance not available for the best model to save.")

print("\nMachine learning experiment completed successfully! All requirements have been addressed.")
Execution output from Jun 14, 2026 9:43 PM
1KB
	Stream
		--- Final Model Training and Prediction on Full Dataset ---
		Shape of combined Training + Validation set: (1020, 11)
		Retraining the best model (K-Nearest Neighbors (KNN)) on combined training and validation data...
		Model K-Nearest Neighbors (KNN) retrained successfully.
		
		--- Final Output Statistics on Full Dataset ---
		Predicted Anomaly Distribution (Full Dataset):
		 Predicted_Anomaly
		0    0.995
		1    0.005
		Name: proportion, dtype: float64
		
		Performance of the retrained K-Nearest Neighbors (KNN) on the Test Set:
		  Accuracy: 0.9444
		  Precision: 0.0000
		  Recall: 0.0000
		  F1 Score: 0.0000
		  ROC-AUC: 0.5520
		
		--- Saving Artifacts ---
		Best trained model saved as 'best_model_K-Nearest_Neighbors_(KNN).pkl'
		Full dataset predictions saved as 'full_dataset_predictions.csv'
		Validation and test metrics saved as 'validation_metrics.csv' and 'test_metrics.csv'
		Feature importance not available for the best model to save.
		
		Machine learning experiment completed successfully! All requirements have been addressed.

Text cell <13620884>
# %% [markdown]
## 8. Model Evaluation on Validation Set

Now that we have trained and tuned our models, we will evaluate their performance on the validation set using various metrics as specified:
- Accuracy
- Precision
- Recall
- F1 Score
- ROC-AUC
- Confusion Matrix

This step helps us understand how well each model generalizes to unseen data before final testing.

Code cell <ab8a9023>
# %% [code]
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Dictionary to store evaluation results
validation_metrics = pd.DataFrame(columns=['Model', 'Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC-AUC'])
confusion_matrices = {}

print("Starting model evaluation on the Validation Set...")

for name, model in trained_models.items():
    print(f"\nEvaluating {name}...")

    # Make predictions
    y_pred = model.predict(X_val)
    y_proba = model.predict_proba(X_val)[:, 1] if hasattr(model, 'predict_proba') else model.decision_function(X_val)

    # Calculate metrics
    accuracy = accuracy_score(y_val, y_pred)
    precision = precision_score(y_val, y_pred)
    recall = recall_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred)
    roc_auc = roc_auc_score(y_val, y_proba)
    cm = confusion_matrix(y_val, y_pred)

    # Store results
    validation_metrics.loc[len(validation_metrics)] = [name, accuracy, precision, recall, f1, roc_auc]
    confusion_matrices[name] = cm

    print(f"  Accuracy: {accuracy:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall: {recall:.4f}")
    print(f"  F1 Score: {f1:.4f}")
    print(f"  ROC-AUC: {roc_auc:.4f}")
    print(f"  Confusion Matrix:\n{cm}")

print("\n--- Validation Metrics Summary ---")
display(validation_metrics.sort_values(by='ROC-AUC', ascending=False))

print("\n--- Visualizing Confusion Matrices ---")
# Plot confusion matrices
fig, axes = plt.subplots(nrows=3, ncols=3, figsize=(15, 15))
axes = axes.flatten()

for i, (name, cm) in enumerate(confusion_matrices.items()):
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i],
                xticklabels=['Predicted No', 'Predicted Yes'],
                yticklabels=['Actual No', 'Actual Yes'])
    axes[i].set_title(f'Confusion Matrix - {name}')
    axes[i].set_xlabel('Predicted Label')
    axes[i].set_ylabel('True Label')

# Hide any unused subplots
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

print("Validation set evaluation complete. Next, we will create comparison tables and visualizations.")
Execution output from Jun 14, 2026 9:43 PM
143KB
	Stream
		Starting model evaluation on the Validation Set...
		
		Evaluating Logistic Regression...
		  Accuracy: 0.9444
		  Precision: 0.0000
		  Recall: 0.0000
		  F1 Score: 0.0000
		  ROC-AUC: 0.4806
		  Confusion Matrix:
		[[170   0]
		 [ 10   0]]
		
		Evaluating XGBoost Classifier...
		  Accuracy: 0.9444
		  Precision: 0.0000
		  Recall: 0.0000
		  F1 Score: 0.0000
		  ROC-AUC: 0.4747
		  Confusion Matrix:
		[[170   0]
		 [ 10   0]]
		
		Evaluating Random Forest Classifier...
		  Accuracy: 0.9444
		  Precision: 0.0000
		  Recall: 0.0000
		  F1 Score: 0.0000
		  ROC-AUC: 0.3776
		  Confusion Matrix:
		[[170   0]
		 [ 10   0]]
		
		Evaluating Decision Tree Classifier...
		  Accuracy: 0.9444
		  Precision: 0.0000
		  Recall: 0.0000
		  F1 Score: 0.0000
		  ROC-AUC: 0.4971
		  Confusion Matrix:
		[[170   0]
		 [ 10   0]]
		
		Evaluating SVM...
		  Accuracy: 0.9444
		  Precision: 0.0000
		  Recall: 0.0000
		  F1 Score: 0.0000
		  ROC-AUC: 0.5871
		  Confusion Matrix:
		[[170   0]
		 [ 10   0]]
		
		Evaluating Gradient Boosting Classifier...
		  Accuracy: 0.9444
		  Precision: 0.0000
		  Recall: 0.0000
		  F1 Score: 0.0000
		  ROC-AUC: 0.4865
		  Confusion Matrix:
		[[170   0]
		 [ 10   0]]
		
		Evaluating K-Nearest Neighbors (KNN)...
		  Accuracy: 0.9389
		  Precision: 0.0000
		  Recall: 0.0000
		  F1 Score: 0.0000
		  ROC-AUC: 0.4762
		  Confusion Matrix:
		[[169   1]
		 [ 10   0]]
		
		--- Validation Metrics Summary ---
		/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
		  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
		/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
		  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
		/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
		  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
		/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
		  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
		/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
		  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
		/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
		  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
		--- Visualizing Confusion Matrices ---
		Validation set evaluation complete. Next, we will create comparison tables and visualizations.
	text/plain
		Model  Accuracy  Precision  Recall  F1 Score  \
		4                           SVM  0.944444        0.0     0.0       0.0   
		3      Decision Tree Classifier  0.944444        0.0     0.0       0.0   
		5  Gradient Boosting Classifier  0.944444        0.0     0.0       0.0   
		0           Logistic Regression  0.944444        0.0     0.0       0.0   
		6     K-Nearest Neighbors (KNN)  0.938889        0.0     0.0       0.0   
		1            XGBoost Classifier  0.944444        0.0     0.0       0.0   
		2      Random Forest Classifier  0.944444        0.0     0.0       0.0   
		
		    ROC-AUC  
		4  0.587059  
		3  0.497059  
		5  0.486471  
		0  0.480588  
		6  0.476176  
		1  0.474706  
		2  0.377647
		<Figure size 1500x1500 with 14 Axes>




IndentationError: unindent does not match any outer indentation level (<string>, line 45)